In [1]:
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import Perceptron, LogisticRegression, SGDClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, PowerTransformer

from src.auxiliares import dataframe_coeficientes
from src.config import DADOS_LIMPOS
from src.graficos import plot_coeficientes, plot_comparar_metricas_modelos
from src.models import (
    RANDOM_STATE, grid_search_cv_classificador, 
    treinar_e_validar_modelo_classificacao, organiza_resultados
)

In [2]:
df = pd.read_parquet(DADOS_LIMPOS)

df.head()

,area_mean,area_se,area_worst,compactness_mean,compactness_se,compactness_worst,concave points_mean,concave points_se,concave points_worst,concavity_mean,...,radius_worst,smoothness_mean,smoothness_se,smoothness_worst,symmetry_mean,symmetry_se,symmetry_worst,texture_mean,texture_se,texture_worst
0,1001.0,153.40,2019.0,0.27760,0.04904,0.6656,0.14710,0.01587,0.2654,0.3001,...,25.38,0.11840,0.006399,0.1622,0.2419,0.03003,0.4601,10.38,0.9053,17.33
1,1326.0,74.08,1956.0,0.07864,0.01308,0.1866,0.07017,0.01340,0.1860,0.0869,...,24.99,0.08474,0.005225,0.1238,0.1812,0.01389,0.2750,17.77,0.7339,23.41
2,1203.0,94.03,1709.0,0.15990,0.04006,0.4245,0.12790,0.02058,0.2430,0.1974,...,23.57,0.10960,0.006150,0.1444,0.2069,0.02250,0.3613,21.25,0.7869,25.53
3,386.1,27.23,567.7,0.28390,0.07458,0.8663,0.10520,0.01867,0.2575,0.2414,...,14.91,0.14250,0.009110,0.2098,0.2597,0.05963,0.6638,20.38,1.1560,26.50
4,1297.0,94.44,1575.0,0.13280,0.02461,0.2050,0.10430,0.01885,0.1625,0.1980,...,22.54,0.10030,0.011490,0.1374,0.1809,0.01756,0.2364,14.34,0.7813,16.67


In [3]:
X = df.drop(columns=['diagnosis'])

y = df['diagnosis']

In [4]:
le = LabelEncoder()

y = le.fit_transform(y)

y[:20]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0])

In [5]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [6]:
preprocessamento = Pipeline(steps=[
    ('scaler', PowerTransformer())
])

In [7]:
classificadores = {
    "DummyClassifier": {
        "preprocessor": None,
        "classificador": DummyClassifier(strategy='stratified')
    },
    "Perceptron": {
        "preprocessor": preprocessamento,
        "classificador": Perceptron()
    },
    "LogisticRegression": {
        "preprocessor": preprocessamento,
        "classificador": LogisticRegression()
    },
    "SGDClassifier": {
        "preprocessor": preprocessamento,
        "classificador": SGDClassifier()
    },
}

In [8]:
resultados = {
    nome_modelo: treinar_e_validar_modelo_classificacao(X, y, kf, **classificador)
    for nome_modelo, classificador in classificadores.items()
}

df_resultados = organiza_resultados(resultados)
df_resultados

,model,fit_time,score_time,test_accuracy,test_balanced_accuracy,test_f1,test_precision,test_recall,test_roc_auc,test_average_precision,time_seconds
0,DummyClassifier,0.0,0.025469,0.473684,0.453652,0.347826,0.326531,0.372093,0.400426,0.347817,0.025469
1,DummyClassifier,0.000998,0.009542,0.552632,0.530789,0.426966,0.413043,0.44186,0.446937,0.356735,0.01054
2,DummyClassifier,0.000997,0.007981,0.596491,0.556548,0.425,0.447368,0.404762,0.583333,0.417544,0.008978
3,DummyClassifier,0.000998,0.010133,0.552632,0.511905,0.37037,0.384615,0.357143,0.574405,0.410618,0.011131
4,DummyClassifier,0.0,0.017245,0.566372,0.533367,0.409639,0.414634,0.404762,0.462944,0.35616,0.017245
5,Perceptron,0.063878,0.012017,0.947368,0.953161,0.933333,0.893617,0.976744,0.992139,0.98945,0.075894
6,Perceptron,0.060877,0.016126,0.964912,0.96266,0.953488,0.953488,0.953488,0.99869,0.997909,0.077003
7,Perceptron,0.067185,0.015622,0.947368,0.943452,0.928571,0.928571,0.928571,0.982474,0.977233,0.082806
8,Perceptron,0.054672,0.015638,0.973684,0.979167,0.965517,0.933333,1.0,1.0,1.0,0.07031
9,Perceptron,0.06999,0.008506,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.078495


In [9]:
df_resultados.groupby("model").mean().sort_values("test_recall")

,fit_time,score_time,test_accuracy,test_balanced_accuracy,test_f1,test_precision,test_recall,test_roc_auc,test_average_precision,time_seconds
model,,,,,,,,,,
DummyClassifier,0.000598,0.014074,0.548362,0.517252,0.39596,0.397238,0.396124,0.493609,0.377775,0.014672
SGDClassifier,0.061345,0.0157,0.964881,0.962406,0.952799,0.955076,0.952824,0.994529,0.99248,0.077045
LogisticRegression,0.073169,0.018323,0.975408,0.970873,0.96636,0.981258,0.952935,0.996632,0.995721,0.091492
Perceptron,0.06332,0.013582,0.966667,0.967688,0.956182,0.941802,0.971761,0.99466,0.992919,0.076902


## GridSearch

In [13]:
param_grid = {
    "clf__C": [0.1, 1,  10, 100],
    "clf__penalty": ['l1', 'l2', 'elasticnet', None],
    "clf__class_weight": [None, "balanced"],
    "clf__l1_ratio" : [0.1, 0.25, 0.5, 0.75, 0.9]
}

In [14]:
clf = LogisticRegression(solver="saga", random_state=RANDOM_STATE)

grid_search = grid_search_cv_classificador(
    clf,
    param_grid, kf, preprocessamento, refit_metric='recall'
)
grid_search

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        Pipeline(steps=[('scaler',
                                                         PowerTransformer())])),
                                       ('clf',
                                        LogisticRegression(random_state=42,
                                                           solver='saga'))]),
             n_jobs=-1,
             param_grid={'clf__C': [0.1, 1, 10, 100],
                         'clf__class_weight': [None, 'balanced'],
                         'clf__l1_ratio': [0.1, 0.25, 0.5, 0.75, 0.9],
                         'clf__penalty': ['l1', 'l2', 'elasticnet', None]},
             refit='recall',
             scoring=['accuracy', 'balanced_accuracy', 'f1', 'precision',
                      'recall', 'roc_auc', 'average_precision'],
             verbose=1)

In [15]:
grid_search.fit(X, y)

Fitting 5 folds for each of 160 candidates, totalling 800 fits


c:\Users\breno\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        Pipeline(steps=[('scaler',
                                                         PowerTransformer())])),
                                       ('clf',
                                        LogisticRegression(random_state=42,
                                                           solver='saga'))]),
             n_jobs=-1,
             param_grid={'clf__C': [0.1, 1, 10, 100],
                         'clf__class_weight': [None, 'balanced'],
                         'clf__l1_ratio': [0.1, 0.25, 0.5, 0.75, 0.9],
                         'clf__penalty': ['l1', 'l2', 'elasticnet', None]},
             refit='recall',
             scoring=['accuracy', 'balanced_accuracy', 'f1', 'precision',
                      'recall', 'roc_auc', 'average_precision'],
             verbose=1)

In [16]:
grid_search.best_params_

{'clf__C': 0.1,
 'clf__class_weight': 'balanced',
 'clf__l1_ratio': 0.1,
 'clf__penalty': 'l2'}

In [17]:
grid_search.best_score_

0.9623477297895903

In [18]:
grid_search.best_estimator_['clf'].coef_

array([[ 0.30776287,  0.51249108,  0.45442998, -0.08335921, -0.33172553,
         0.07880288,  0.49038811,  0.13461129,  0.45020865,  0.42494767,
         0.05462929,  0.44867428, -0.18393305, -0.25154422,  0.12127361,
         0.28292901,  0.30080053,  0.39246367,  0.28248957,  0.43920334,
         0.42714151,  0.21077149, -0.04788555,  0.44371669,  0.1529948 ,
        -0.34752493,  0.3884476 ,  0.51214069, -0.0474521 ,  0.60874479]])